In [5]:
df=pd.read_csv('my_inventory_data.csv')
df['unit_cost'] = df['sell_price'] * 0.6
print(df)

  item_id  demand  sell_price  holding_cost_per_unit  shortage_cost_per_unit  \
0    SP01     100         500                     10                      50   
1    SP02     150         300                      5                      30   
2    SP03      80        1000                     20                     100   
3    SP04     200         150                      2                      15   
4    SP05      50        2000                     50                     200   

   unit_cost  
0      300.0  
1      180.0  
2      600.0  
3       90.0  
4     1200.0  


In [6]:
import sys
import os

# Thêm thư mục src vào hệ thống để import
sys.path.append(os.path.abspath("../src"))

from budget_constraint import solve_bounded_knapsack

In [12]:
import numpy as np
import pandas as pd

n_items = 500

# Bước 1: Tạo các thành phần cơ bản trước
sell_prices = np.random.uniform(50, 100, n_items)
demands = np.random.randint(1, 50, n_items)

# Bước 2: Định nghĩa data sử dụng các biến đã tạo
data = {
    'item_id': [f'PROD_{i}' for i in range(n_items)],
    'sell_price': sell_prices,
    'demand': demands,
    # unit_cost = 60% giá bán
    'unit_cost': 0.6 * sell_prices,
    # net_value = giá bán * nhu cầu (Tổng doanh thu mục tiêu)
    'net_value_per_unit': sell_prices * 0.4
}

df = pd.DataFrame(data)

In [15]:
# --- PHẦN 2: THIẾT LẬP NGÂN SÁCH THỬ NGHIỆM ---
total_budget = 500000 # 50 triệu đơn vị

# --- PHẦN 3: GỌI HÀM VÀ IN KẾT QUẢ ---
print("🚀 Đang chạy thử thuật toán tối ưu hóa...")
total_val, df_final = solve_bounded_knapsack(df, total_budget)

print("-" * 50)
print(f"💰 TỔNG GIÁ TRỊ TỐI ƯU (TEST): {total_val:,.2f}")
print(f"📦 Số lượng mã hàng cần nhập: {len(df_final)}")
print("-" * 50)

# Hiển thị 10 món đầu tiên trong kế hoạch nhập hàng
if not df_final.empty:
    display(df_final.head(10))
else:
    print("Ngân sách thử nghiệm quá thấp!")

🚀 Đang chạy thử thuật toán tối ưu hóa...
--------------------------------------------------
💰 TỔNG GIÁ TRỊ TỐI ƯU (TEST): 333,333.33
📦 Số lượng mã hàng cần nhập: 452
--------------------------------------------------


,item_id,quantity_to_buy,unit_cost,total_cost,total_value
0,PROD_0,25,55.169920,1379.247996,919.498664
1,PROD_1,38,30.342540,1153.016501,768.677667
2,PROD_2,3,51.110993,153.332980,102.221987
3,PROD_4,26,43.129840,1121.375836,747.583891
4,PROD_5,30,37.049202,1111.476064,740.984043
5,PROD_6,13,51.146130,664.899694,443.266463
6,PROD_7,13,54.513844,708.679975,472.453317
7,PROD_8,2,46.392909,92.785819,61.857213
8,PROD_10,49,31.550062,1545.953021,1030.635347
9,PROD_11,38,45.143868,1715.466982,1143.644655


## Thực hiện tối ưu trên tập dữ liệu M5

In [14]:
import pandas as pd
import sys
import os

# Thêm thư mục src vào hệ thống để import
sys.path.append(os.path.abspath("../../src"))

from budget_constraint import solve_bounded_knapsack

In [2]:
simulation_summary_with_error = pd.read_csv("../../dataset/result/simulation/simulation_summary_with_forecast_error.csv")
simulation_summary_with_error.columns

Index(['Unnamed: 0', 'review_period', 'lead_time', 'service_level_target',
       'z_value', 'mu_error', 'std_error', 'safety_stock', 'initial_inventory',
       'num_periods_simulated', 'num_orders', 'total_demand', 'total_sales',
       'total_shortage_units', 'fill_rate', 'cycle_service_level_empirical',
       'daily_service_level_empirical', 'average_ending_inventory',
       'total_order_qty', 'total_holding_cost', 'total_shortage_cost',
       'total_order_cost', 'total_cost', 'id', 'item_id', 'dept_id',
       'store_id', 'cat_id', 'state_id', 'unit_price', 'holding_cost_per_unit',
       'shortage_cost_per_unit', 'estimation_window', 'simulation_window'],
      dtype='str')

In [7]:
total_sales = simulation_summary_with_error['unit_price'] * simulation_summary_with_error['total_order_qty'] 
print(total_sales.sum() * 0.6)
print(total_sales.sum() * 0.4)


1805353.0038729273
1203568.6692486182


In [10]:
df_optimize = simulation_summary_with_error[['id', 'unit_price', 'total_order_qty']]
df_optimize['unit_cost'] = df_optimize['unit_price'] * 0.6
df_optimize['net_value_per_unit'] = df_optimize['unit_price'] * 0.4
df_optimize.rename(columns={'id': 'item_id'}, inplace=True)
df_optimize.rename(columns={'total_order_qty': 'demand'}, inplace=True)
df_optimize.head()

,item_id,unit_price,demand,unit_cost,net_value_per_unit
0,HOUSEHOLD_2_229_TX_1,5.94,6.869842,3.564,2.376
1,HOBBIES_1_339_CA_2,7.98,26.361800,4.788,3.192
2,FOODS_2_073_CA_4,5.28,3.643477,3.168,2.112
3,HOUSEHOLD_1_238_TX_3,2.48,39.884554,1.488,0.992
4,HOUSEHOLD_2_144_TX_1,7.58,2.189923,4.548,3.032


In [23]:
import numpy as np

budget_range = np.arange(0, 1800000, 20000)

results = []

for budget in budget_range:
    print(f"Running optimization with budget = {budget}")
    
    total_val, df_final = solve_bounded_knapsack(df_optimize, budget)
    
    # Tổng hợp metrics
    summary = {
        "budget": budget,
        "total_value": total_val,
    }
    
    results.append(summary)

df_summary = pd.DataFrame(results)
df_summary

Running optimization with budget = 0
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/vietha/venv/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/f2fe653c1e8c42aab6c6bf2b0c075884-pulp.mps -max -sec 60 -timeMode elapsed -branch -printingOptions all -solution /tmp/f2fe653c1e8c42aab6c6bf2b0c075884-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 30496 COLUMNS
At line 182947 RHS
At line 213439 BOUNDS
At line 243930 ENDATA
Problem MODEL has 30491 rows, 30490 columns and 60980 elements
Coin0008I MODEL read with 0 errors
seconds was changed from 1e+100 to 60
Option for timeMode changed from cpu to elapsed
Continuous objective value is 0 - 0.06 seconds
Cgl0004I processed model has 0 rows, 0 columns (0 integer (0 of which binary)) and 0 elements
Cbc3007W No integer variables - nothing to do
Cuts at root node changed objective from 0 to -1.79769e+308
Probing was tried 0 times and created 0

,budget,total_value
0,0,0.000
1,20000,13333.332
2,40000,26666.664
3,60000,40000.000
4,80000,53333.332
...,...,...
85,1700000,1133333.332
86,1720000,1146666.664
87,1740000,1160000.000
88,1760000,1173333.332


In [24]:
df_summary.to_csv("../../dataset/result/optimization_summary.csv", index=False)